# Quick start

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/quick_start.ipynb)

You call every method in Andrey the same way: pass it your data, and read the result it returns.
This notebook loads the protein measurements of Sachs et al. (2005), learns a graph from them
with four classic methods, PC, GES, FCI, and DirectLiNGAM, reads each result, and draws each
graph.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.17.0` from PyPI;
local kernels use the installed environment.

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[viz]==0.17.0" pandas

In [2]:
import andrey
import andrey.viz
from andrey.data import load_dataset

## The data

Sachs and colleagues (2005) measured 11 proteins and phospholipids in 853 single human immune
cells. Biologists had already mapped how these molecules signal to one another, so a graph
learned from the measurements can be checked against a known network of 17 links.
`load_dataset` downloads the measurements on first use and caches them.

Each row is one cell and each column one molecule. For your own data, pass a pandas DataFrame
or a NumPy array of the same shape: one row per sample, one column per variable.

In [3]:
sachs = load_dataset("sachs")  # 853 cells, 11 proteins, and the known network
data = sachs.data
data.head()

,raf,mek,plc,pip2,pip3,erk,akt,pka,pkc,p38,jnk
0,26.4,13.2,8.82,18.30,58.80,6.61,17.0,414.0,17.00,44.9,40.0
1,35.9,16.5,12.30,16.80,8.13,18.60,32.5,352.0,3.37,16.5,61.5
2,59.4,44.1,14.60,10.20,13.00,14.90,32.5,403.0,11.40,31.9,19.5
3,73.0,82.8,23.10,13.50,1.29,5.83,11.8,528.0,13.70,28.6,23.1
4,33.7,19.8,5.19,9.73,24.80,21.10,46.1,305.0,4.66,25.7,81.3


## In math

Each column of `data` is a variable $X_j$, here $d = 11$ of them, and each row is one draw from a
distribution $P$ over $V = \{X_1, \dots, X_d\}$. We model each variable as a function of its
parents in a directed acyclic graph (DAG) $G$, its direct causes, and of its own noise:

$$X_j = f_j(X_{\mathrm{pa}_G(j)}, N_j), \qquad j = 1, \dots, d,$$

with the noise terms $N_j$ jointly independent, so no hidden variable causes two columns. A method
estimates $G$, or the part of $G$ the data identify: PC and GES below return a class of DAGs, FCI
drops the independence of the noise to allow hidden common causes, and DirectLiNGAM identifies $G$
itself by assuming linear $f_j$ and non-Gaussian noise. The
[concepts page](https://andrey.abel.ai/docs/guides/concepts.html) states each assumption and what
each graph claims.

## PC

PC starts with every pair of molecules linked. It removes a link when a statistical test finds
the two independent, either alone or given some of their other neighbors, and orients a link
only when a pattern in the tests forces its direction.

In [4]:
pc = andrey.pc(data)
print(pc)

pc  cpdag  |  11 nodes  |  8 edges (6 undirected, 2 directed)

  raf -- mek
  plc -- pip3
  pip2 -- pip3
  erk -- akt
  erk -- pka
  akt -- pka
  p38 -> pkc
  jnk -> pkc

  indep_test: fisherz
  tie_break: rank


The first line names the method and the kind of graph. This one is a CPDAG: it stands for
every causal graph that independence tests cannot tell apart. A directed edge (`->`) points
the same way in all of them, and an undirected edge (`--`) can point either way.

You can read the same parts in code. `pc.structure` is the graph: its `kind` says how to read
the edges, and `oriented_edges()` lists each edge as (source, target, type), with your column
names. `pc.metadata` records the settings of the run.

In [5]:
graph = pc.structure
print("kind:", graph.kind)
print("directed:", [edge for edge in graph.oriented_edges() if edge[2] == "directed"])
print("test:", pc.metadata["indep_test"])

kind: cpdag
directed: [('p38', 'pkc', 'directed'), ('jnk', 'pkc', 'directed')]
test: fisherz


`andrey.viz.draw` renders the graph as an SVG, which the notebook shows. A directed edge ends
in an arrowhead; an undirected edge has none and is drawn in amber.

In [6]:
andrey.viz.draw(pc.structure)

'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 360 478" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="360" height="478" fill="#fafbfc"/><line x1="93.00" y1="445.91" x2="167.00" y2="445.91" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="90.64" y1="282.11" x2="169.36" y2="337.38" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="92.93" y1="354.11" x2="167.07" y2="346.23" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="92.74" y1="191.21" x2="167.26" y2="176.15" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="92.98" y1="194.48" x2="267.02" y2="203.74" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="192.42" y1="177.41" x2="267.58" y2="200.59" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="158.68" y1="63.89" x2="92.05" y2="36.96" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="167.95,67.64 160.55,59.25 156.80,68.53" fill="#3a4658" stroke-linejoin="round"/><line x1="158.68" y1="81.13" x2="92.05" y2="108.06" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="167.95,77.38 156.80,76.50 160.55,85.77" fill="#3a4658" stroke-linejoin="round"/><circle cx="80.00" cy="445.91" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="470.91" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">raf</text><circle cx="180.00" cy="445.91" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="180.00" y="470.91" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">mek</text><circle cx="80.00" cy="274.64" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="299.64" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">plc</text><circle cx="80.00" cy="355.49" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="380.49" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">pip2</text><circle cx="180.00" cy="344.85" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="180.00" y="369.85" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">pip3</text><circle cx="80.00" cy="193.79" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="218.79" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">erk</text><circle cx="180.00" cy="173.57" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="180.00" y="198.57" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">akt</text><circle cx="280.00" cy="204.43" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="280.00" y="229.43" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">pka</text><circle cx="180.00" cy="72.51" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="180.00" y="97.51" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">pkc</text><circle cx="80.00" cy="32.09" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="57.09" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" strok

## GES

GES, greedy equivalence search, reaches a graph a different way: it searches for the graph
with the best score, instead of testing independence. You call it the same way.

In [7]:
ges = andrey.ges(data)
print(ges)

ges  cpdag  |  11 nodes  |  8 edges (6 undirected, 2 directed)

  raf -- mek
  plc -- pip3
  pip2 -- pip3
  erk -- akt
  erk -- pka
  akt -- pka
  p38 -> pkc
  jnk -> pkc

  score: 6.692e+04
  score_func: bic
  penalty_discount: 1
  ebic_gamma: 1
  tie_break: rank


GES returns a CPDAG too, and on these data it is the same graph as PC's. Its metadata holds
the score of the graph, where lower is better, and the settings that produced it.

In [8]:
print("same edges as PC:", ges.structure.oriented_edges() == pc.structure.oriented_edges())
andrey.viz.draw(ges.structure)

same edges as PC: True


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 360 478" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="360" height="478" fill="#fafbfc"/><line x1="93.00" y1="445.91" x2="167.00" y2="445.91" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="90.64" y1="282.11" x2="169.36" y2="337.38" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="92.93" y1="354.11" x2="167.07" y2="346.23" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="92.74" y1="191.21" x2="167.26" y2="176.15" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="92.98" y1="194.48" x2="267.02" y2="203.74" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="192.42" y1="177.41" x2="267.58" y2="200.59" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="158.68" y1="63.89" x2="92.05" y2="36.96" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="167.95,67.64 160.55,59.25 156.80,68.53" fill="#3a4658" stroke-linejoin="round"/><line x1="158.68" y1="81.13" x2="92.05" y2="108.06" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="167.95,77.38 156.80,76.50 160.55,85.77" fill="#3a4658" stroke-linejoin="round"/><circle cx="80.00" cy="445.91" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="470.91" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">raf</text><circle cx="180.00" cy="445.91" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="180.00" y="470.91" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">mek</text><circle cx="80.00" cy="274.64" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="299.64" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">plc</text><circle cx="80.00" cy="355.49" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="380.49" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">pip2</text><circle cx="180.00" cy="344.85" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="180.00" y="369.85" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">pip3</text><circle cx="80.00" cy="193.79" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="218.79" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">erk</text><circle cx="180.00" cy="173.57" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="180.00" y="198.57" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">akt</text><circle cx="280.00" cy="204.43" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="280.00" y="229.43" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">pka</text><circle cx="180.00" cy="72.51" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="180.00" y="97.51" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">pkc</text><circle cx="80.00" cy="32.09" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="57.09" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" strok

## FCI

PC and GES assume that every common cause of two measured variables is measured too. FCI does
not: it allows hidden common causes, and says less about each edge in return.

In [9]:
fci = andrey.fci(data)
print(fci)

fci  pag  |  11 nodes  |  8 edges (6 circle, 2 partially_directed)

  raf o-o mek
  plc o-o pip3
  pip2 o-o pip3
  erk o-o akt
  erk o-o pka
  akt o-o pka
  p38 o-> pkc
  jnk o-> pkc

  indep_test: fisherz
  tie_break: rank


FCI returns a PAG. A circle (`o`) at an end of an edge means the tests do not say whether that
end is an arrowhead or a tail. So `p38 o-> pkc` says that pkc does not cause p38, and leaves
open whether p38 causes pkc or a hidden variable causes both. The drawing shows each circle as
a hollow circle.

In [10]:
andrey.viz.draw(fci.structure)

'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 360 498" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="360" height="498" fill="#fafbfc"/><line x1="101.40" y1="466.02" x2="158.60" y2="466.02" stroke="#7c5cd8" stroke-width="2.0" stroke-linecap="round"/><circle cx="97.20" cy="466.02" r="4.20" fill="#ffffff" stroke="#7c5cd8" stroke-width="2.0"/><circle cx="162.80" cy="466.02" r="4.20" fill="#ffffff" stroke="#7c5cd8" stroke-width="2.0"/><line x1="97.51" y1="307.04" x2="162.49" y2="352.66" stroke="#7c5cd8" stroke-width="2.0" stroke-linecap="round"/><circle cx="94.08" cy="304.63" r="4.20" fill="#ffffff" stroke="#7c5cd8" stroke-width="2.0"/><circle cx="165.92" cy="355.07" r="4.20" fill="#ffffff" stroke="#7c5cd8" stroke-width="2.0"/><line x1="101.28" y1="373.33" x2="158.72" y2="367.22" stroke="#7c5cd8" stroke-width="2.0" stroke-linecap="round"/><circle cx="97.10" cy="373.78" r="4.20" fill="#ffffff" stroke="#7c5cd8" stroke-width="2.0"/><circle cx="162.90" cy="366.78" r="4.20" fill="#ffffff" stroke="#7c5cd8" stroke-width="2.0"/><line x1="100.98" y1="209.65" x2="159.02" y2="197.92" stroke="#7c5cd8" stroke-width="2.0" stroke-linecap="round"/><circle cx="96.86" cy="210.49" r="4.20" fill="#ffffff" stroke="#7c5cd8" stroke-width="2.0"/><circle cx="163.14" cy="197.09" r="4.20" fill="#ffffff" stroke="#7c5cd8" stroke-width="2.0"/><line x1="101.37" y1="215.03" x2="258.63" y2="223.40" stroke="#7c5cd8" stroke-width="2.0" stroke-linecap="round"/><circle cx="97.18" cy="214.81" r="4.20" fill="#ffffff" stroke="#7c5cd8" stroke-width="2.0"/><circle cx="262.82" cy="223.62" r="4.20" fill="#ffffff" stroke="#7c5cd8" stroke-width="2.0"/><line x1="200.45" y1="199.99" x2="259.55" y2="218.22" stroke="#7c5cd8" stroke-width="2.0" stroke-linecap="round"/><circle cx="196.44" cy="198.75" r="4.20" fill="#ffffff" stroke="#7c5cd8" stroke-width="2.0"/><circle cx="263.56" cy="219.46" r="4.20" fill="#ffffff" stroke="#7c5cd8" stroke-width="2.0"/><line x1="99.67" y1="80.69" x2="161.70" y2="43.07" stroke="#7c5cd8" stroke-width="2.0" stroke-linecap="round"/><polygon points="91.12,85.88 102.26,84.97 97.07,76.42" fill="#7c5cd8" stroke-linejoin="round"/><circle cx="165.29" cy="40.90" r="4.20" fill="#ffffff" stroke="#7c5cd8" stroke-width="2.0"/><line x1="102.54" y1="97.17" x2="159.02" y2="108.59" stroke="#7c5cd8" stroke-width="2.0" stroke-linecap="round"/><polygon points="92.74,95.19 101.55,102.07 103.53,92.27" fill="#7c5cd8" stroke-linejoin="round"/><circle cx="163.14" cy="109.42" r="4.20" fill="#ffffff" stroke="#7c5cd8" stroke-width="2.0"/><circle cx="80.00" cy="466.02" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="491.02" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">raf</text><circle cx="180.00" cy="466.02" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="180.00" y="491.02" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">mek</text><circle cx="80.00" cy="294.74" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="319.74" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">plc</text><circle cx="80.00" cy="375.60" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="400.60" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">pip2</text><circle cx="180.00" cy="364.96" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="180.00" y="389.96" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">pip3</text><circle cx="80.00" cy="213.89" r="13.00" fi

## DirectLiNGAM

DirectLiNGAM assumes that each variable is a linear function of its causes plus noise that is
not Gaussian. When that holds, the data identify one graph, so it returns a DAG: every edge has
a direction.

In [11]:
lingam = andrey.direct_lingam(data)
print(lingam)

direct_lingam  dag  |  11 nodes  |  11 edges (11 directed)

  mek -> raf  (+1.2)
  mek -> pip2  (+0.285)
  plc -> pip2  (+0.583)
  plc -> pip3  (+0.236)
  pip2 -> pip3  (+0.0952)
  erk -> akt  (+1.37)
  erk -> pka  (+1.84)
  pka -> akt  (+0.017)
  pkc -> p38  (+1.28)
  jnk -> pkc  (-0.0526)
  jnk -> p38  (+0.0582)

  order: erk -> plc -> mek -> pip2 -> pip3 -> pka -> jnk -> akt -> pkc -> p38 -> raf

  tie_break: rank


The number beside an edge is its weight, the coefficient of the cause in the linear equation
for the effect, and `order` is the causal order the method found. These directions rest on the
method's assumption, so treat them as hypotheses to test.

In [12]:
andrey.viz.draw(lingam.structure)

'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 360 446" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="360" height="446" fill="#fafbfc"/><line x1="157.29" y1="290.99" x2="92.84" y2="301.27" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="167.16,289.41 156.50,286.05 158.08,295.92" fill="#3a4658" stroke-linejoin="round"/><line x1="90.91" y1="310.40" x2="160.71" y2="355.69" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="169.09,361.14 163.43,351.50 157.98,359.89" fill="#3a4658" stroke-linejoin="round"/><line x1="91.82" y1="408.55" x2="159.08" y2="377.78" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="168.18,373.62 157.00,373.23 161.16,382.33" fill="#3a4658" stroke-linejoin="round"/><line x1="92.96" y1="412.92" x2="257.07" y2="399.83" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="267.04,399.03 256.68,394.85 257.47,404.81" fill="#3a4658" stroke-linejoin="round"/><line x1="192.46" y1="371.92" x2="257.96" y2="391.43" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="267.54,394.29 259.38,386.64 256.53,396.23" fill="#3a4658" stroke-linejoin="round"/><line x1="92.95" y1="175.55" x2="257.09" y2="160.71" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="267.05,159.81 256.64,155.73 257.54,165.69" fill="#3a4658" stroke-linejoin="round"/><line x1="92.46" y1="180.43" x2="157.96" y2="199.94" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="167.54,202.80 159.38,195.15 156.53,204.74" fill="#3a4658" stroke-linejoin="round"/><line x1="259.25" y1="168.57" x2="191.73" y2="200.90" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="268.27,164.25 257.10,164.06 261.41,173.08" fill="#3a4658" stroke-linejoin="round"/><line x1="192.22" y1="36.46" x2="258.37" y2="60.39" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="267.78,63.79 260.07,55.69 256.67,65.09" fill="#3a4658" stroke-linejoin="round"/><line x1="158.02" y1="38.82" x2="92.42" y2="59.06" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="167.58,35.87 156.55,34.05 159.50,43.60" fill="#3a4658" stroke-linejoin="round"/><line x1="257.01" y1="67.60" x2="93.00" y2="63.24" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="267.00,67.87 257.14,62.60 256.88,72.60" fill="#3a4658" stroke-linejoin="round"/><circle cx="180.00" cy="287.36" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="180.00" y="312.36" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">raf</text><circle cx="80.00" cy="303.32" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="328.32" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">mek</text><circle cx="80.00" cy="413.96" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="438.96" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">plc</text><circle cx="180.00" cy="368.21" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="180.00" y="393.21" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">pip2</text><circle cx="280.00" cy="398.00" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="280.00" y="423.00" text-anchor="middle" font-size="11" font-weight="600" fill="#5c6675" stroke="#fafbfc" stroke-width="3" stroke-linejoin="round" paint-order="stroke">pip3</text><circle cx="80.00" cy="176.72" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.

## Next

The [Methods notebooks](https://andrey.abel.ai/docs/examples/index.html#methods) each take one
family of methods further: its other methods, its settings, and the data it suits.